In [1]:
import polars as pl
import numpy as np

In [2]:
data_base_path = "../data"

## Import Original Dataset

In [3]:
movies_data = pl.read_parquet(f"{data_base_path}/movies_plots_dataset.parquet")

In [4]:
movies_data.head()

Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot
i64,str,str,str,str,str,str,str
1901,"""Kansas Saloon Smashers""","""American""","""Unknown""",null,"""unknown""","""https://en.wikipedia.org/wiki/…","""A bartender is working at a sa…"
1901,"""Love by the Light of the Moon""","""American""","""Unknown""",null,"""unknown""","""https://en.wikipedia.org/wiki/…","""The moon, painted with a smili…"
1901,"""The Martyred Presidents""","""American""","""Unknown""",null,"""unknown""","""https://en.wikipedia.org/wiki/…","""The film, just over a minute l…"
1901,"""Terrible Teddy, the Grizzly Ki…","""American""","""Unknown""",null,"""unknown""","""https://en.wikipedia.org/wiki/…","""Lasting just 61 seconds and co…"
1902,"""Jack and the Beanstalk""","""American""","""George S. Fleming, Edwin S. Po…",null,"""unknown""","""https://en.wikipedia.org/wiki/…","""The earliest known adaptation …"


## Calculating and Adding Plots Embeddings to Dataset

In [5]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("all-MiniLM-L6-v2")
plot_vectors = encoder.encode(movies_data["Plot"])

/Users/luca/work/xtream/amld/amld-2025-ace-of-splades/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TypeError: cannot select elements using key of type 'int64': np.int64(26064)

In [ ]:
movies_data = movies_data.with_columns(
    pl.Series(name="vector", values=plot_vectors),
)

In [ ]:
movies_data.write_parquet(f"{data_base_path}/movies_plots_dataset_embd_minilm.parquet")

## Create LanceDB Table

In [ ]:
import lancedb

uri = f"{data_base_path}/movies_embeddings"
db = lancedb.connect(uri)

In [ ]:
movies_table = db.create_table("movies", movies_data, exist_ok=True)

In [ ]:
movies_table.create_fts_index("Title", use_tantivy=False)
movies_table.create_fts_index("Cast", use_tantivy=False)

## Query Tests

In [ ]:
query = "Star Wars" 
query_vector = encoder.encode(query)

In [ ]:
movies_table.search(query_vector).limit(10).select(['Title', 'Director', 'Plot']).to_list()

In [ ]:
query
(
    movies_table.search(query_type="hybrid")
    .vector(query_vector)
    .text(query)
    .limit(20).select(['Title', 'Director']).to_list()
)